# vepyr — VEP Annotation

Annotate variants from a VCF file using a pre-built Ensembl VEP parquet cache.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import vepyr

In [ ]:
# Configure paths
VCF = "/path/to/input.vcf.gz"
CACHE_DIR = "/path/to/parquet/115_GRCh38_vep"  # from vepyr.build_cache()
REFERENCE_FASTA = "/path/to/Homo_sapiens.GRCh38.dna.primary_assembly.fa"

## Full annotation (--everything)

Enables all annotation features: HGVS, allele frequencies, co-located variants,
SIFT/PolyPhen predictions, regulatory features, etc. Produces 97 columns
with typed Arrow output (Float32 for frequencies, List\<Utf8\> for multi-valued fields).

In [ ]:
lf = vepyr.annotate(
    VCF,
    CACHE_DIR,
    everything=True,
    reference_fasta=REFERENCE_FASTA,
)
lf

In [ ]:
df = lf.collect()
print(f"Variants: {df.height:,}")
print(f"Columns: {df.width}")
df.head(5)

## Inspect annotation columns

In [ ]:
# Key annotation columns
df.select([
    "chrom", "start", "ref", "alt",
    "most_severe_consequence",
    "SYMBOL", "Gene", "IMPACT",
    "HGVSc", "HGVSp",
    "SIFT", "PolyPhen",
]).head(10)

In [ ]:
# Allele frequencies
df.select([
    "chrom", "start", "ref", "alt",
    "AF", "gnomADe_AF", "gnomADg_AF", "MAX_AF",
]).head(10)

In [ ]:
# Consequence distribution
df.group_by("most_severe_consequence").len().sort("len", descending=True).head(15)

## Selective annotation

Instead of `everything=True`, you can pick individual flags:

In [ ]:
# HGVS + allele frequencies only
lf_selective = vepyr.annotate(
    VCF,
    CACHE_DIR,
    hgvs=True,
    af=True,
    af_gnomadg=True,
    reference_fasta=REFERENCE_FASTA,
)
lf_selective.collect().head(5)

## Filter and export

In [ ]:
import polars as pl

# Filter to high-impact variants
high_impact = (
    lf
    .filter(pl.col("IMPACT") == "HIGH")
    .select(["chrom", "start", "ref", "alt", "SYMBOL", "most_severe_consequence", "HGVSc", "HGVSp"])
    .collect()
)
print(f"High-impact variants: {high_impact.height}")
high_impact.head(10)

In [ ]:
# Export to parquet
# df.write_parquet("/tmp/annotated.parquet")

# Export to CSV
# df.write_csv("/tmp/annotated.csv")